In [1]:
import pandas as pd
data = pd.read_csv(r"combined_dataset.csv")

In [2]:
data.shape

(685671, 94)

In [5]:
# Chuyển đổi timestamp sang datetime
start_dt = pd.to_datetime(data['timestamp_start'])
end_dt = pd.to_datetime(data['timestamp_end'])

# Tính duration
duration = (end_dt - start_dt).dt.total_seconds()

# Tạo dataframe mới chỉ chứa các mẫu có duration 1.0 giây
data_1s = data[duration == 1.0].reset_index(drop=True)

# Hiển thị số lượng
print("=== TRƯỚC KHI LỌC ===")
print(f"Số lượng sample: {len(data)}")
print(data['label2'].value_counts())

print("\n=== SAU KHI LỌC (Chỉ lấy 1s) ===")
print(f"Số lượng sample: {len(data_1s)}")
print(data_1s['label2'].value_counts())

=== TRƯỚC KHI LỌC ===
Số lượng sample: 685671
label2
benign        400672
recon         105848
dos            57736
ddos           56692
mitm           25490
malware        24177
web             9040
bruteforce      6016
Name: count, dtype: int64

=== SAU KHI LỌC (Chỉ lấy 1s) ===
Số lượng sample: 227191
label2
benign        136800
recon          33648
dos            18420
ddos           18056
mitm            8062
malware         7541
web             2796
bruteforce      1868
Name: count, dtype: int64


In [9]:
data_1s["label2"].value_counts()

label2
benign        136800
recon          33648
dos            18420
ddos           18056
mitm            8062
malware         7541
web             2796
bruteforce      1868
Name: count, dtype: int64

In [10]:
del data

In [11]:
data = data_1s

In [12]:
del data_1s

1. Imputation: Check for missing values and imput them

In [13]:
print(data.isnull().sum().sort_values(ascending=False)[:50])
# không còn cột nào có giá trị null, không cần phải imputation

device_name                            0
device_mac                             0
label_full                             0
label1                                 0
label2                                 0
label3                                 0
label4                                 0
timestamp                              0
timestamp_start                        0
timestamp_end                          0
log_data-ranges_avg                    0
log_data-ranges_max                    0
log_data-ranges_min                    0
log_data-ranges_std_deviation          0
log_data-types                         0
log_data-types_count                   0
log_interval-messages                  0
log_messages_count                     0
network_fragmentation-score            0
network_fragmented-packets             0
network_header-length_avg              0
network_header-length_max              0
network_header-length_min              0
network_header-length_std_deviation    0
network_interval

2. Reduce memory usage: Check for data types and convert them to more efficient ones if possible

In [14]:
def reduce_memory_usage(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
            print(f"Column {col} converted to float32")
        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
            print(f"Column {col} converted to int32")
    return df

In [15]:
data = reduce_memory_usage(data)

Column log_data-ranges_avg converted to float32
Column log_data-ranges_max converted to float32
Column log_data-ranges_min converted to float32
Column log_data-ranges_std_deviation converted to float32
Column log_data-types_count converted to int32
Column log_interval-messages converted to float32
Column log_messages_count converted to int32
Column network_fragmentation-score converted to float32
Column network_fragmented-packets converted to int32
Column network_header-length_avg converted to float32
Column network_header-length_max converted to float32
Column network_header-length_min converted to float32
Column network_header-length_std_deviation converted to float32
Column network_interval-packets converted to float32
Column network_ip-flags_avg converted to float32
Column network_ip-flags_max converted to float32
Column network_ip-flags_min converted to float32
Column network_ip-flags_std_deviation converted to float32
Column network_ip-length_avg converted to float32
Column netwo

3. One-hot encoding: 

In [17]:
# in ra các cột object
print(data.select_dtypes(include=['object']).columns)

Index(['device_name', 'device_mac', 'label_full', 'label1', 'label2', 'label3',
       'label4', 'timestamp', 'timestamp_start', 'timestamp_end',
       'log_data-types', 'network_ips_all', 'network_ips_dst',
       'network_ips_src', 'network_macs_all', 'network_macs_dst',
       'network_macs_src', 'network_ports_all', 'network_ports_dst',
       'network_ports_src', 'network_protocols_all', 'network_protocols_dst',
       'network_protocols_src'],
      dtype='object')


In [18]:
label_to_drop = ["label_full", "label1", "label3","label4"]
data.drop(columns=label_to_drop, inplace=True)

In [25]:
cols_to_drop = ["device_name","device_mac","timestamp","timestamp_end","network_ips_all","network_macs_all","network_macs_dst","network_macs_src","network_ports_all","network_protocols_all"]

In [26]:
data.drop(columns=cols_to_drop, inplace=True)

In [27]:
print(data.select_dtypes(include=['object']).columns)

Index(['label2', 'timestamp_start', 'log_data-types', 'network_ips_dst',
       'network_ips_src', 'network_ports_dst', 'network_ports_src',
       'network_protocols_dst', 'network_protocols_src'],
      dtype='object')


In [28]:
print(data["log_data-types"].value_counts())

log_data-types
[]                       178639
['numeric']               37739
['array', 'numeric']       5311
['string', 'array']        4892
['string', 'numeric']       525
['array']                    57
['string']                   28
Name: count, dtype: int64


In [31]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer


# ==========================================
# PIPELINE XỬ LÝ
# ==========================================

# Bước 1: Parse (Chuyển đổi) chuỗi thành List thực sự
def parse_string_to_list(val):
    try:
        # ast.literal_eval dịch chuỗi "['a', 'b']" thành cấu trúc list ['a', 'b'] của Python
        if isinstance(val, str):
            return ast.literal_eval(val)
        return val if isinstance(val, list) else []
    except (ValueError, SyntaxError):
        # Trả về list rỗng nếu gặp giá trị lỗi/NaN
        return []

# Áp dụng hàm parse tạo một cột tạm
data['parsed_list'] = data['log_data-types'].apply(parse_string_to_list)


# Bước 2: Khởi tạo và áp dụng MultiLabelBinarizer
mlb = MultiLabelBinarizer()

# fit_transform sẽ quét qua các list và tạo ra ma trận nhị phân (0 và 1)
binary_matrix = mlb.fit_transform(data['parsed_list'])

# Lấy tên các nhãn (classes) mà mô hình tìm thấy (array, numeric, string)
# Thêm tiền tố 'log_type_' để tên cột rõ nghĩa hơn khi đưa vào mô hình học máy
new_column_names = [f"log_type_{c}" for c in mlb.classes_]

# Biến ma trận thành DataFrame
df_binary = pd.DataFrame(binary_matrix, columns=new_column_names, index=data.index)


# Bước 3: Nối dữ liệu mới vào bảng gốc và dọn dẹp
df_final = pd.concat([data, df_binary], axis=1)

# Xóa bỏ cột chuỗi gốc và cột list tạm thời
df_final = df_final.drop(columns=['log_data-types', 'parsed_list'])

print("--- DỮ LIỆU SAU KHI XỬ LÝ ---")
print(df_final)

--- DỮ LIỆU SAU KHI XỬ LÝ ---
        label2              timestamp_start  log_data-ranges_avg  \
0         ddos  2025-01-23T15:31:10.709000Z                  0.0   
1         ddos  2025-01-23T15:31:11.709000Z                  0.0   
2         ddos  2025-01-23T15:31:12.709000Z                  0.0   
3         ddos  2025-01-23T15:31:13.709000Z                  0.0   
4         ddos  2025-01-23T15:31:14.709000Z                  0.0   
...        ...                          ...                  ...   
227186  benign  2025-09-09T15:09:35.400000Z                  0.0   
227187  benign  2025-09-09T15:09:36.400000Z                  0.0   
227188  benign  2025-09-09T15:09:37.400000Z                  0.0   
227189  benign  2025-09-09T15:09:38.400000Z                  0.0   
227190  benign  2025-09-09T15:09:39.400000Z                  0.0   

        log_data-ranges_max  log_data-ranges_min  \
0                       0.0                  0.0   
1                       0.0                  0.0 

In [33]:
print(df_final.select_dtypes(include=['object']).columns)

Index(['label2', 'timestamp_start', 'network_ips_dst', 'network_ips_src',
       'network_ports_dst', 'network_ports_src', 'network_protocols_dst',
       'network_protocols_src'],
      dtype='object')


In [42]:
def process_protocol_column(df, col_name, prefix='proto', threshold=100):
    print(f"Bắt đầu xử lý cột: {col_name}")
    # Parse chuỗi thành list (sử dụng hàm parse_string_to_list bạn đã định nghĩa ở cell trước)
    parsed_series = df[col_name].apply(parse_string_to_list)
    
    # Đếm tần suất xuất hiện của từng phần tử trong các list
    all_elements = [item for sublist in parsed_series for item in sublist]
    counts = pd.Series(all_elements).value_counts()
    
    # Tập hợp các giao thức xuất hiện > threshold
    frequent_protos = set(counts[counts > threshold].index)
    print(f" - Số giao thức > {threshold} lần: {len(frequent_protos)}")
    
    # Hàm gom nhóm
    def group_protocols(proto_list):
        res = set()
        for p in proto_list:
            if p in frequent_protos:
                res.add(p)
            else:
                res.add('other')
        return list(res)
        
    # Áp dụng hàm gom nhóm
    grouped_series = parsed_series.apply(group_protocols)
    
    # Sử dụng MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    binary_matrix = mlb.fit_transform(grouped_series)
    
    # Tạo DataFrame mới với tên cột thêm prefix
    new_cols = [f"{prefix}_{c}" for c in mlb.classes_]
    df_binary = pd.DataFrame(binary_matrix, columns=new_cols, index=df.index)
    
    # Nối vào bảng gốc và Drop cột dữ liệu chuỗi ban đầu đi
    df_out = pd.concat([df, df_binary], axis=1).drop(columns=[col_name])
    print(f" - Hoàn tất! Tạo ra {len(new_cols)} cột mới: {new_cols[:5]} ...")
    return df_out

# ================================
# ÁP DỤNG HÀM VÀO CÁC CỘT CỦA BẠN
# ================================
if 'network_protocols_src' in df_final.columns:
    df_final = process_protocol_column(df_final, 'network_protocols_src', prefix='src_proto', threshold=100)

if 'network_protocols_dst' in df_final.columns:
    df_final = process_protocol_column(df_final, 'network_protocols_dst', prefix='dst_proto', threshold=100)

print("\n--- Kiểm tra lại các cột Object còn sót lại ---")
print(df_final.select_dtypes(include=['object']).columns)
print("\nKích thước df_final mới:", df_final.shape)

Bắt đầu xử lý cột: network_protocols_src
 - Số giao thức > 100 lần: 22
 - Hoàn tất! Tạo ra 23 cột mới: ['src_proto_arp', 'src_proto_data', 'src_proto_data-text-lines', 'src_proto_dhcpv6', 'src_proto_dns'] ...
Bắt đầu xử lý cột: network_protocols_dst
 - Số giao thức > 100 lần: 35
 - Hoàn tất! Tạo ra 36 cột mới: ['dst_proto_arp', 'dst_proto_c1222', 'dst_proto_chargen', 'dst_proto_data', 'dst_proto_data-text-lines'] ...

--- Kiểm tra lại các cột Object còn sót lại ---
Index(['label2', 'timestamp_start', 'network_ips_dst', 'network_ips_src',
       'network_ports_dst', 'network_ports_src'],
      dtype='object')

Kích thước df_final mới: (227191, 139)


In [83]:
print(df_final.select_dtypes(include=['object']).columns)

Index(['label', 'timestamp', 'network_ips_dst', 'network_ips_src',
       'network_ports_dst', 'network_ports_src'],
      dtype='object')


In [93]:
print(df_final["label"].head(100000).value_counts())

label
recon         33648
dos           18420
ddos          18056
benign         9609
mitm           8062
malware        7541
web            2796
bruteforce     1868
Name: count, dtype: int64
